# Ranking evaluation: Qini curves, AUUC, and a sleeping-dogs check

Notebook 03 evaluated each CATE model with a decile table. Two more checks belong here:

1. **Qini curve / AUUC** - a whole-ranking summary instead of ten separate bins.
2. **Sleeping dogs** - are there users the ads actively *hurt*? (Question 5.)

**Qini curve.** Sort held-out users by predicted CATE, highest first. At each cutoff `k`, take the top `k` users and measure the *actual* treatment-vs-control difference among them, times `k`:

```
qini(k) = (mean outcome treated - mean outcome control, among top k) * k
```

That's the incremental outcomes you'd have gained by targeting only those `k` users. The dashed **random** line goes from 0 to the overall ATE x N - what you get by targeting in arbitrary order. A good ranking bows above that line. **AUUC** here is the area between the two (the Qini coefficient): positive means the ranking beats random.

Like the decile table, this uses only held-out data and only measured treatment/control differences - no per-user ground truth is needed or used.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import pandas as pd

from uplift.data import load_sample, split_train_eval
from uplift.cate_models import (
    fit_s_learner, predict_s_learner,
    fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner,
    fit_dr_learner, predict_dr_learner,
    fit_causal_forest, predict_causal_forest,
)
from uplift.evaluation import qini_curve, auuc, decile_table

df = load_sample()
train, eval_ = split_train_eval(df)


def predict_all(outcome):
    return {
        "S-learner": predict_s_learner(fit_s_learner(train, outcome), eval_),
        "T-learner": predict_t_learner(fit_t_learner(train, outcome), eval_),
        "X-learner": predict_x_learner(fit_x_learner(train, outcome), eval_),
        "DR-learner": predict_dr_learner(fit_dr_learner(train, outcome), eval_),
        "Causal forest": predict_causal_forest(fit_causal_forest(train, outcome, n_estimators=100), eval_),
    }


cates = {outcome: predict_all(outcome) for outcome in ["visit", "conversion"]}

## Qini curves and AUUC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, outcome in zip(axes, ["visit", "conversion"]):
    rows = []
    for name, cate in cates[outcome].items():
        q = qini_curve(eval_, cate, outcome)
        ax.plot(q["fraction"], q["qini"], label=name)
        rows.append({"model": name, "AUUC": auuc(q),
                     "qini_at_10pct": q.loc[(q["fraction"] - 0.1).abs().idxmin(), "qini"],
                     "qini_at_100pct": q["qini"].iloc[-1]})
    ax.plot(q["fraction"], q["random"], "k--", label="random targeting")
    ax.set_title(outcome)
    ax.set_xlabel("fraction of users targeted (highest predicted CATE first)")
    ax.set_ylabel("incremental outcomes (eval set)")
    ax.legend()
    print(f"=== {outcome} ===")
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print()

plt.tight_layout()
plt.show()

## Sleeping dogs: does any group show a negative effect?

The natural place to look is the *bottom* of each model's ranking - the users it thinks are least helped or most hurt. For the bottom two deciles of each model, report the measured ATE with a **Bonferroni-corrected** CI (`alpha = 0.05 / 10`, matching notebook 05). A sleeping-dogs finding needs `ci_high < 0`: an interval entirely below zero.

Selection here is legitimate: deciles are chosen by *predicted* CATE from models fit on `train`, and the effect is measured on `eval_`, so no eval-set outcome is used to choose where to look.

In [ ]:
for outcome in ["visit", "conversion"]:
    print(f"=== {outcome}: bottom two deciles, 99.5% CI ===")
    for name, cate in cates[outcome].items():
        tbl = decile_table(eval_, cate, outcome, alpha=0.05 / 10).head(2)
        for _, r in tbl.iterrows():
            flag = "  <-- CI entirely below zero" if r["ci_high"] < 0 else ""
            print(f"{name:14s} d{int(r['decile'])}  ate={r['actual_ate']:+.5f}  [{r['ci_low']:+.5f}, {r['ci_high']:+.5f}]{flag}")
    print()

## Interpretation

**Qini / AUUC**

- Every model beats random targeting on both outcomes; the top 10% of users by predicted CATE captures roughly half of the total incremental outcomes (about 1,360-1,560 of 2,854 for `visit`; about 140-300 of 437 for `conversion`). This is the same "top decile is real" story as the decile tables, now from a whole-curve view.
- **The S-learner is not uniformly bad.** On `visit` it has the *highest* AUUC (~973 vs ~830-880 for the others). On `conversion` it collapses (~10 vs ~85-120), consistent with the all-zeros predictions seen in notebook 03. Its shrinkage problem depends on how rare the outcome is.
- **Do not rank models on these AUUC numbers.** They are single point estimates from one train/eval split with no uncertainty attached. The gaps among the four non-S models (and, for `visit`, the S-learner too) are small relative to the noise that made the middle deciles unreliable in notebook 03. An honest statement is "all comparable, none clearly best"; a bootstrap over the eval set would be needed to say more.
- The `conversion` curves are lumpier than `visit`'s: 0.2% base rate means the curve moves on a few hundred events.

**Sleeping dogs**

- **No decile in any model has a CI entirely below zero, for either outcome.** There is no evidence that ads hurt any identifiable group.
- That is *absence of evidence*, not evidence of absence. Bottom-decile CIs are wide (for `visit` about +/-0.01, comparable to the whole ATE of 0.011), so a modest negative effect could be hiding inside them. The data rules out large harm in these bins, not small harm.
- Some bottom deciles are even positive (e.g. causal forest `conversion` d1 is significantly above zero), so the lowest *predicted* CATE is not the lowest *true* one - another reminder that the ranking is only reliable at the top.
- Caveat on interpretation: the features are anonymized and randomly projected, so even a real sleeping-dogs segment couldn't be described in terms of what kind of user it is.